# 🧩 Mini-Lab: Typed Prompt Parser

**Module: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Represent** prompts, roles, and parameters as versioned dataclass structures
2. **Serialize** prompt configurations to JSON for storage and version tracking
3. **Parse** model responses defensively, handling malformed or missing JSON gracefully
4. **Explain** why treating prompts and responses as structured data matters for production reliability

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Prompt Versioning | Prompt versioning is the practice of tracking, labeling, and safely changing prompts over time so you can reproduce outputs, compare improvements, and roll back quickly when a prompt update causes errors or security issues. |
| Output Parsing | Output parsing is turning a model’s response into reliable, structured data (like JSON fields) your code can safely use,while detecting mistakes, missing info, or prompt-injection tricks in the output. |
| JSON Serialization | Records as text: JSON stores structured data as readable key value pairs, and JSON Lines (JSONL) stores one JSON object per line for streams and large datasets. |
| Dataclass Shape | Use a function for behavior and a dataclass for shape: a named, typed bundle of fields that makes pipeline inputs and outputs explicit. Use a class only when you truly need both state and behavior together. |


## Recap

In **mini-define-ai-protocol**, we built `AIRequest` and `AIResponse` dataclasses that flow across the interface seam. Those shapes hold a single prompt string. They don't capture the **full prompt configuration**: the system message, the version, and the parameters that produced a specific result. If you can't reproduce *exactly* which prompt version generated a response, debugging is impossible. And on the output side, LLMs don't always return clean JSON, so your code must parse defensively.

## Step 1 — The Problem with Loose Strings

Here's how prompts often look in quick prototypes — scattered strings with no structure or version tracking.

In [2]:
# ❌ The loose-string approach — no structure, no versioning

system_msg = "You are a support classifier. Return one category."
user_template = "Classify this ticket: {ticket}. Categories: {categories}"
model = "gpt-4o-mini"
temp = 0.0

# A week later someone edits the prompt. Which version produced yesterday's result?
print("Loose prompt has no version and no way to reproduce past results.")

Loose prompt has no version and no way to reproduce past results.


## Step 2 — Prompts as Versioned Data

Let's define a `PromptConfig` dataclass that bundles everything together — the template, the role, the parameters, and crucially, a **version string**.

> 🧩 **Prompt Versioning.** The tracking practice from the video, now applied in code.

In [3]:
from dataclasses import dataclass, field, asdict
from typing import Optional
import json


@dataclass
class PromptConfig:
    """A versioned, structured prompt configuration."""
    name: str                           # e.g. "ticket-classifier"
    version: str                        # e.g. "1.2"
    system: str                         # system message
    user_template: str                  # user message with {placeholders}
    temperature: float = 0.0
    max_tokens: int = 256
    model: str = "gpt-4o-mini"

    def render(self, **kwargs) -> str:
        """Fill in the template placeholders."""
        return self.user_template.format(**kwargs)


# Create version 1.0 of our classifier prompt
v1 = PromptConfig(
    name="ticket-classifier",
    version="1.0",
    system="You are a support ticket classifier. Return exactly one category.",
    user_template="Classify this ticket: {ticket}\nCategories: {categories}\nCategory:",
)

# Render it with actual values
rendered = v1.render(ticket="I was charged twice", categories="billing, technical, account, other")
print(f"Config: {v1.name} v{v1.version}")
print(f"Rendered user message:\n{rendered}")

Config: ticket-classifier v1.0
Rendered user message:
Classify this ticket: I was charged twice
Categories: billing, technical, account, other
Category:


## Step 3 — Serialize to JSON for Version Control

Dataclasses convert to dicts with `asdict()`, which means they serialize to JSON trivially. You can save prompt configs to files, commit them to git, and load the exact version that produced any past result.

In [4]:
# Serialize to a JSON string (ready to save to a file and commit to git)
config_json = json.dumps(asdict(v1), indent=2)
print("Serialized keys:", list(asdict(v1).keys()))

Serialized keys: ['name', 'version', 'system', 'user_template', 'temperature', 'max_tokens', 'model']


## Step 4 — Evolving Prompts with Version Tracking

When you improve a prompt, bump the version. Now you can always trace which config produced which output.

In [5]:
# Version 2: improved prompt with stricter instructions
v2 = PromptConfig(
    name="ticket-classifier",
    version="2.0",
    system="You are a senior support agent. Classify the following ticket into exactly one category.",
    user_template=(
        "Ticket: {ticket}\n"
        "Allowed categories: {categories}\n\n"
        "Respond with ONLY the category name, nothing else."
    ),
    temperature=0.0,
)

print(f"Built {v2.name} v{v2.version} (bumped from v{v1.version})")

Built ticket-classifier v2.0 (bumped from v1.0)


In [6]:
# Render v2 to see the difference
print(v2.render(ticket="I was charged twice", categories="billing, technical, account, other"))

Ticket: I was charged twice
Allowed categories: billing, technical, account, other

Respond with ONLY the category name, nothing else.


## Step 5 — The Response Parsing Problem

Now let's tackle the **output** side. You ask the LLM for JSON, but what you actually get can be... surprising. Let's look at real-world response shapes that break naive parsing.

> 🧩 **Output Parsing.** The idea from the video, tested on real messy responses.

In [7]:
# Simulated LLM responses — only ONE is clean JSON
llm_responses = [
    # ✅ Clean JSON
    '{"category": "billing", "confidence": 0.95}',

    # ❌ Wrapped in markdown code fences (very common!)
    '```json\n{"category": "technical", "confidence": 0.88}\n```',

    # ❌ Has preamble text before the JSON
    'Sure! Here is the classification:\n{"category": "account", "confidence": 0.72}',

    # ❌ Truncated JSON (hit max_tokens) — no valid JSON to recover
    '{"category": "billing", "confi',
]

# Naive parsing: json.loads() works on the clean one...
print("Clean:", json.loads(llm_responses[0]))

# ...but crashes on the fenced one
try:
    json.loads(llm_responses[1])
except json.JSONDecodeError as e:
    print("Fenced: ❌ JSONDecodeError —", e.args[0][:50])

Clean: {'category': 'billing', 'confidence': 0.95}
Fenced: ❌ JSONDecodeError — Expecting value: line 1 column 1 (char 0)


## Step 6 — Defensive Response Parsing

A defensive parser handles the common failure modes: markdown fences, preamble text, invalid JSON, and missing fields. Instead of crashing, it returns a structured result with an error flag.

In [8]:
import re


@dataclass
class ParsedResponse:
    """Result of defensive parsing — always safe to use."""
    data: dict = field(default_factory=dict)
    raw: str = ""
    ok: bool = False
    error: Optional[str] = None


def parse_llm_json(raw_text: str) -> ParsedResponse:
    """Defensively parse JSON from an LLM response."""
    raw_text = raw_text.strip()

    # 1. Strip markdown code fences if present
    fence_match = re.search(r"```(?:json)?\s*\n?(.*?)\n?```", raw_text, re.DOTALL)
    if fence_match:
        raw_text = fence_match.group(1).strip()

    # 2. Try a direct parse
    try:
        return ParsedResponse(data=json.loads(raw_text), raw=raw_text, ok=True)
    except json.JSONDecodeError:
        pass

    # 3. Look for the first {...} block inside surrounding text
    brace_match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if brace_match:
        try:
            return ParsedResponse(data=json.loads(brace_match.group()), raw=brace_match.group(), ok=True)
        except json.JSONDecodeError:
            pass

    # 4. Nothing worked: return a structured failure instead of crashing
    return ParsedResponse(raw=raw_text, ok=False, error="Could not extract valid JSON from response")

In [9]:
# Fenced and preamble responses now parse
print("Fenced:  ", parse_llm_json(llm_responses[1]).data)
print("Preamble:", parse_llm_json(llm_responses[2]).data)

# Truncated JSON fails gracefully with an error flag
bad = parse_llm_json(llm_responses[3])
print("Truncated:", bad.ok, "-", bad.error)

Fenced:   {'category': 'technical', 'confidence': 0.88}
Preamble: {'category': 'account', 'confidence': 0.72}
Truncated: False - Could not extract valid JSON from response


## Step 7 — Safe Field Extraction

Even when parsing succeeds, the expected fields might be missing or have wrong types. Always extract with defaults.

In [10]:
def extract_classification(parsed: ParsedResponse) -> dict:
    """Extract category and confidence with safe defaults."""
    if not parsed.ok:
        return {"category": "other", "confidence": 0.0, "parse_error": parsed.error}

    category = parsed.data.get("category", "other")
    confidence = parsed.data.get("confidence", 0.0)

    # Wrong types fall back to defaults instead of crashing later
    if not isinstance(category, str):
        category = "other"
    if not isinstance(confidence, (int, float)):
        confidence = 0.0

    return {"category": category.lower().strip(), "confidence": float(confidence)}


# A parsed response gives real data. A failed one degrades to "other" with zero confidence.
print("OK:    ", extract_classification(parse_llm_json(llm_responses[2])))
print("Failed:", extract_classification(parse_llm_json(llm_responses[3])))

OK:     {'category': 'account', 'confidence': 0.72}
Failed: {'category': 'other', 'confidence': 0.0, 'parse_error': 'Could not extract valid JSON from response'}


## Step 8 — Putting It Together: Versioned In, Defensive Out

Let's combine both halves — the structured prompt config and the defensive parser — into a single pipeline that an `AIClient` (from the previous lab) would sit between.

In [11]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Simulate the full flow: config → render → (AI would go here) → parse
config = v2  # using our versioned prompt config

# Render the prompt
rendered_prompt = config.render(
    ticket="My login keeps failing with error 403",
    categories="billing, technical, account, other",
)

# Simulate LLM response (in production, this comes from ai_client.complete())
simulated_response = '```json\n{"category": "technical", "confidence": 0.91}\n```'

# Defensive parse
parsed = parse_llm_json(simulated_response)
result = extract_classification(parsed)

summary = f"""
## Full Pipeline Trace

| Stage | Value |
|-------|-------|
| **Prompt config** | `{config.name}` v`{config.version}` |
| **System message** | {config.system} |
| **Rendered prompt** | {rendered_prompt[:60]}... |
| **Raw LLM response** | `{simulated_response[:50]}...` |
| **Parse OK?** | {parsed.ok} |
| **Extracted category** | `{result['category']}` |
| **Confidence** | `{result['confidence']}` |

Every piece is **traceable** — you know which prompt version produced which result,
and the response was parsed defensively rather than blindly trusted.
"""

md(summary)


## Full Pipeline Trace

| Stage | Value |
|-------|-------|
| **Prompt config** | `ticket-classifier` v`2.0` |
| **System message** | You are a senior support agent. Classify the following ticket into exactly one category. |
| **Rendered prompt** | Ticket: My login keeps failing with error 403
Allowed catego... |
| **Raw LLM response** | ````json
{"category": "technical", "confidence": 0....` |
| **Parse OK?** | True |
| **Extracted category** | `technical` |
| **Confidence** | `0.91` |

Every piece is **traceable** — you know which prompt version produced which result,
and the response was parsed defensively rather than blindly trusted.


## 🎯 Summary

### Key Takeaways

1. **Prompts as versioned data** — wrapping prompts, system messages, and parameters in a `PromptConfig` dataclass with a version string makes every prompt reproducible, diffable, and traceable
2. **JSON serialization** — `dataclasses.asdict()` + `json.dumps()` lets you save prompt configs to files or version control with zero extra code
3. **Defensive parsing** — LLM responses often contain markdown fences, preamble text, or truncated JSON; a layered parser (strip fences → try direct parse → search for braces → fail gracefully) handles real-world messiness
4. **Safe field extraction** — even after successful JSON parsing, always use `.get()` with defaults and type checks to prevent downstream crashes from missing or mistyped fields